# Complete Guide to Neural Network Optimizers

## Understanding Optimizers

An optimizer is an algorithm that updates the weights during training using the gradients computed from backpropagation. Different optimizers have different strategies for how to use gradients to update weights.

### Why Different Optimizers?

```
Basic Gradient Descent:
w_new = w_old - learning_rate × gradient

Problems with basic approach:
1. Single learning rate for all weights
   - Some weights need different step sizes
   - One learning rate doesn't fit all situations

2. Constant learning rate throughout training
   - Early: need large steps (far from optimum)
   - Later: need small steps (fine-tuning)
   - Fixed rate doesn't adapt

3. Gets stuck easily
   - Local minima trap
   - Saddle points
   - Plateaus

4. Oscillates around minimum
   - Doesn't smoothly converge
   - Takes many iterations

Solutions: Advanced optimizers!
- Adapt learning rate
- Remember past gradients
- Use momentum
- Handle different scales
```

---

## 1. SGD (Stochastic Gradient Descent)

### What is SGD?

SGD is the basic gradient descent algorithm using mini-batches. It's the foundation for all modern optimizers.

```
Formula:
w_new = w_old - learning_rate × gradient

Where gradient computed on mini-batch (not single sample, not all data)
```

### How SGD Works:

```
1. Select random mini-batch
2. Compute gradient on batch
3. Update all weights using same learning rate
4. Move to next batch
5. Repeat

Characteristics:
- Same learning rate for all weights
- No momentum
- No adaptation
- Simple but effective
```

### SGD Example:

Let me trace through training with SGD:

**Network Setup:**
```
Simple network: 1 input → 2 neurons → 1 output
Current weights:
W1 = [0.5, -0.3]ᵀ
W2 = [0.8, 0.6]
Learning rate = 0.1
Training data: x = 2, y = 5 (target)
```

**Iteration 1:**

Forward Pass:
```
z1 = W1 × x = [0.5, -0.3]ᵀ × 2 = [1.0, -0.6]ᵀ
a1 = ReLU(z1) = [1.0, 0]ᵀ (second neuron dead)

z2 = W2 × a1 = [0.8, 0.6] × [1.0, 0]ᵀ = 0.8
ŷ = 0.8

Loss = (5 - 0.8)² = 17.64
```

Backward Pass (Compute Gradients):
```
∂Loss/∂ŷ = -2(5 - 0.8) = -8.4

∂Loss/∂W2 = ∂Loss/∂ŷ × a1 = -8.4 × [1.0, 0]ᵀ = [-8.4, 0]ᵀ

∂Loss/∂W1[0] = -8.4 × W2[0] × ReLU'(z1[0]) × x
              = -8.4 × 0.8 × 1 × 2
              = -13.44

∂Loss/∂W1[1] = 0 (dead neuron)
```

Weight Update (SGD):
```
W2_new = W2 - learning_rate × ∂Loss/∂W2
       = [0.8, 0.6] - 0.1 × [-8.4, 0]ᵀ
       = [0.8 + 0.84, 0.6 - 0]
       = [1.64, 0.6]

W1[0]_new = 0.5 - 0.1 × (-13.44)
          = 0.5 + 1.344
          = 1.844

W1[1]_new = -0.3 (no update, gradient = 0)
```

**Iteration 2:**

Forward Pass with new weights:
```
z1 = [1.844, -0.3]ᵀ × 2 = [3.688, -0.6]ᵀ
a1 = ReLU(z1) = [3.688, 0]ᵀ

z2 = [1.64, 0.6] × [3.688, 0]ᵀ = 6.049
ŷ = 6.049

Loss = (5 - 6.049)² = 1.099
```

Improvement! Loss went from 17.64 → 1.099

Backward Pass:
```
∂Loss/∂ŷ = -2(5 - 6.049) = 2.098

∂Loss/∂W2 = 2.098 × [3.688, 0]ᵀ = [7.743, 0]ᵀ

∂Loss/∂W1[0] = 2.098 × 1.64 × 1 × 2 = 6.881
```

Weight Update:
```
W2_new = [1.64, 0.6] - 0.1 × [7.743, 0]ᵀ
       = [0.866, 0.6]

W1[0]_new = 1.844 - 0.1 × 6.881
          = 1.156
```

**Pattern Observation:**

```
Iteration 1:
  Gradient: -13.44 for W1[0]
  Update: +1.344 (opposite direction)
  Loss: 17.64 → 1.099

Iteration 2:
  Gradient: +6.881 for W1[0]
  Update: -0.688 (opposite direction)
  Loss: 1.099 → smaller

The weights oscillate around optimal value!
This is typical with constant learning rate.
```

### SGD Characteristics:

```
Pros:
✓ Simple to understand and implement
✓ Works reasonably well
✓ Good baseline
✓ Low memory overhead

Cons:
✗ Constant learning rate (not adaptive)
✗ Can oscillate around minimum
✗ Slower convergence than modern optimizers
✗ Same step size for all weights
✗ Gets stuck in local minima

Oscillation Problem:
Loss
  |*
  | *\*/*\/*
  |   \ / \ /
  |    X   X
  |   / \ / \
  |_/_____\___ Epoch
  
  Notice the zigzag pattern!
  Takes longer to converge.
```

### When to Use SGD:

```
Use SGD when:
- Learning simple models
- Understand basic optimization
- Computational resources limited
- Don't want complex optimizer

Don't use when:
- Training deep networks (use Adam instead)
- Need faster convergence
- Have complex loss landscapes
```

---

## 2. SGD with Momentum

### What is Momentum?

Momentum addresses the oscillation problem. It accumulates past gradients, creating "inertia" that helps the optimizer move smoothly through valleys.

**Analogy:**
```
Gradient Descent without momentum: Rolling a ball down a hill
- Ball bounces around
- Changes direction with every slope change
- Slow and erratic

Gradient Descent with momentum: Rolling a heavy ball
- Momentum carries it forward
- Doesn't change direction abruptly
- Smoother, faster descent
- Builds up speed on consistent slopes
```

### How Momentum Works:

```
Traditional SGD:
w_new = w - lr × gradient

SGD with Momentum:
1. Keep "velocity" vector (accumulated gradients)
2. Update velocity: v = β × v - lr × gradient
   (β is momentum factor, typically 0.9)
3. Update weight: w = w + v

Effect:
- Velocity accumulates in consistent direction
- Dampens oscillations
- Accelerates convergence
```

### SGD with Momentum Example:

**Setup:**
```
Same network as before
Momentum β = 0.9
Learning rate = 0.1
Initial velocity for all weights: 0
```

**Iteration 1:**

```
Gradient for W1[0]: -13.44 (computed as before)

Velocity update:
v = β × v_old - lr × gradient
v = 0.9 × 0 - 0.1 × (-13.44)
v = 1.344

Weight update:
W1[0]_new = W1[0] + v
          = 0.5 + 1.344
          = 1.844

Same as SGD without momentum (because v_old = 0)
```

**Iteration 2:**

```
Gradient for W1[0]: +6.881 (computed as before)

Velocity update:
v = β × v_old - lr × gradient
v = 0.9 × 1.344 - 0.1 × 6.881
v = 1.210 - 0.688
v = 0.522

Weight update:
W1[0]_new = 1.844 + 0.522
          = 2.366

Compare to SGD without momentum:
SGD: W1[0] = 1.844 - 0.1 × 6.881 = 1.156 (moved backward)
Momentum: W1[0] = 1.844 + 0.522 = 2.366 (continued forward!)

With momentum:
- Previous velocity was +1.344 (moving forward)
- New gradient says to move backward (-6.881)
- But old velocity is strong (×0.9 = 1.210)
- Net result: still move forward (+0.522)
- Smoother path!
```

**Iteration 3:**

```
Gradient for W1[0]: +3.2 (suppose)

Velocity update:
v = 0.9 × 0.522 - 0.1 × 3.2
v = 0.470 - 0.32
v = 0.15

Weight update:
W1[0]_new = 2.366 + 0.15 = 2.516

Velocity decreases as we approach optimum!
Eventually: gradient = 0, velocity decays to 0
Smooth convergence!
```

### Visualization: Momentum vs No Momentum

```
Loss Landscape (2D):

Without Momentum (Oscillates):
    *
   /|\
  / | \
 /  |  \
    ▼
Path zigzags across valley!
Reaches bottom: very wavy
Time: many iterations

    *
     \
      \*
       \
        \*
         \___

With Momentum (Smooth):
    *
   /
  /
 /____
       \
        \___
         ___▼

Path goes straight down valley!
Reaches bottom: smooth diagonal
Time: fewer iterations

The momentum carries through oscillations!
```

### Momentum Effect on Convergence:

```
Loss Curve Comparison:

Without Momentum:
Loss
  |*
  | *\*/*\/*\/*
  |   \  /  \  /
  |    \/    \/
  |_____________ Epoch
  
  Oscillates! Very wavy.

With Momentum (β=0.9):
Loss
  |*
  | *
  |  *
  |   *
  |    ***
  |_________ Epoch
  
  Smooth descent! Much better!

Convergence: 2-3x faster with momentum!
```

### Momentum Hyperparameter (β):

```
β = momentum factor (typically 0.9, sometimes 0.99)

β = 0 (no momentum):
v_new = 0 × v_old - lr × gradient = -lr × gradient
Effect: Back to basic SGD
No accumulation

β = 0.9 (default):
v_new = 0.9 × v_old - lr × gradient
Effect: 90% of old velocity + new gradient
Good balance: remembers past, responsive to changes

β = 0.99 (strong momentum):
v_new = 0.99 × v_old - lr × gradient
Effect: Very strong inertia
Better for consistent gradients
Might overshoot near minimum

Typical: β = 0.9 is sweet spot
```

### When to Use Momentum:

```
Use SGD with Momentum when:
✓ Training moderate depth networks
✓ Want faster convergence than basic SGD
✓ Have relatively simple loss landscape
✓ Training small to medium models

Still consider Adam or RMSProp for:
✗ Deep networks
✗ Complex loss landscapes
✗ When adaptive learning rate helps
```

---

## 3. Nesterov Momentum

### What is Nesterov Momentum?

Nesterov Momentum is an improvement over standard momentum. Instead of computing gradient at current position, it computes gradient at look-ahead position (where momentum would take us).

**Intuition:**
```
Standard Momentum: "Slide downhill, building momentum"
Nesterov: "Look ahead to where momentum takes us, then adjust"

Analogy: 
Walking downhill with momentum
Standard: Feel slope at current position
Nesterov: Look at slope of where you'll land, adjust accordingly
```

### How Nesterov Works:

```
Standard Momentum:
v = β × v - lr × gradient(current_position)
w = w + v

Nesterov Momentum:
1. Look ahead: look_ahead = w + β × v
2. Compute gradient at look-ahead position
3. Update velocity: v = β × v - lr × gradient(look_ahead)
4. Update weight: w = w + v

Effect:
- Computes gradient "after" momentum
- Anticipates overshooting
- Corrects preemptively
- Smoother, more efficient
```

### Nesterov Example:

**Setup:**
```
Same network
Momentum β = 0.9
Learning rate = 0.1
W1[0] = 0.5
Initial velocity: 0
```

**Iteration 1:**

Standard Momentum:
```
v = 0.9 × 0 - 0.1 × (-13.44) = 1.344
W1[0] = 0.5 + 1.344 = 1.844
```

Nesterov:
```
Look ahead: 
look_ahead = 0.5 + 0.9 × 0 = 0.5 (no velocity yet)

Compute gradient at look_ahead position:
Gradient still -13.44 (first iteration)

v = 0.9 × 0 - 0.1 × (-13.44) = 1.344
W1[0] = 0.5 + 1.344 = 1.844

Same in first iteration (velocity = 0)
```

**Iteration 2:**

Standard Momentum:
```
Current position: W1[0] = 1.844
Velocity: v = 1.344

Gradient at current position: +6.881

v_new = 0.9 × 1.344 - 0.1 × 6.881
      = 1.210 - 0.688
      = 0.522

W1[0] = 1.844 + 0.522 = 2.366
```

Nesterov:
```
Current position: W1[0] = 1.844
Velocity: v = 1.344

Look ahead:
look_ahead = 1.844 + 0.9 × 1.344
           = 1.844 + 1.210
           = 3.054

Compute gradient at look_ahead position:
Suppose gradient at 3.054 is: +4.2 (different!)

v_new = 0.9 × 1.344 - 0.1 × 4.2
      = 1.210 - 0.42
      = 0.79

W1[0] = 1.844 + 0.79 = 2.634

Difference from standard: 2.366 vs 2.634
Nesterov made a different update because it looked ahead!
```

### Why Nesterov is Better:

```
Standard Momentum Problem:
- Uses old gradient information
- Gradient from position we're leaving
- Might overshoot optimum
- Builds up too much speed

Nesterov Solution:
- Looks ahead with accumulated momentum
- Computes gradient at future position
- Corrects course early
- Prevents overshoot

Convergence:
- Slightly faster than standard momentum
- More stable near optimum
- Better for well-conditioned problems

Loss Curve:
Standard momentum can oscillate:
    * (might overshoot)
     \*
      \*_ (overshoots)
       \  * (comes back)
        \_*

Nesterov prevents overshoot:
    *
     \*
      \*
       \*
        \*_ (reaches and stops smoothly)
```

### When to Use Nesterov:

```
Use Nesterov when:
✓ Want slightly better convergence than momentum
✓ Training convex or nearly-convex problems
✓ Don't have adaptive learning rates

Most frameworks default to:
- Adam (adaptive, modern default)
- SGD with Momentum (simple, reliable)
- Nesterov (less common now, but good)
```

---

## 4. Adagrad (Adaptive Gradient)

### What is Adagrad?

Adagrad adapts the learning rate for each weight based on historical gradients. Weights with large gradients get smaller steps; weights with small gradients get larger steps.

**Key Idea:**
```
Problem with constant learning rate:
- Gradient of w1 is usually 5
- Gradient of w2 is usually 0.1
- Same learning rate for both?
- w1 overshoots, w2 barely moves

Solution (Adagrad):
- Accumulate squared gradients
- Divide learning rate by square root of accumulated
- w1 with large gradients: smaller step
- w2 with small gradients: larger step
- Adaptive!
```

### How Adagrad Works:

```
1. Keep running sum of squared gradients: G
2. For each weight: G += gradient²
3. Update weight: w = w - (lr / √(G + ε)) × gradient

Effect:
- Weights with large gradients: smaller steps (√G large)
- Weights with small gradients: larger steps (√G small)
- Self-balancing learning rates
- Each weight has own effective learning rate

ε: small constant (e.g., 1e-8) to prevent division by zero
```

### Adagrad Example:

**Setup:**
```
Two weights in output layer: W_out = [0.8, 0.6]
Learning rate: 0.1
Initial G (accumulated squared gradients): [0, 0]
ε = 1e-8
```

**Iteration 1:**

Gradients:
```
∂Loss/∂W_out[0] = 5.0 (large)
∂Loss/∂W_out[1] = 0.2 (small)
```

Accumulate squared gradients:
```
G[0] += (5.0)² = 0 + 25 = 25
G[1] += (0.2)² = 0 + 0.04 = 0.04

G = [25, 0.04]
```

Effective learning rates:
```
lr_eff[0] = 0.1 / √(25 + 1e-8) = 0.1 / 5.0 = 0.02
lr_eff[1] = 0.1 / √(0.04 + 1e-8) = 0.1 / 0.2 = 0.5

w1 gets lr 0.02 (small)
w2 gets lr 0.5 (large)
Adaptive!
```

Weight updates:
```
W_out[0] = 0.8 - 0.02 × 5.0 = 0.8 - 0.1 = 0.7
W_out[1] = 0.6 - 0.5 × 0.2 = 0.6 - 0.1 = 0.5

Interesting: both update by 0.1!
But through different mechanisms:
- w1: large gradient, small lr
- w2: small gradient, large lr
```

**Iteration 2:**

Gradients:
```
∂Loss/∂W_out[0] = 4.5 (still large)
∂Loss/∂W_out[1] = 0.15 (still small)
```

Accumulate squared gradients:
```
G[0] += (4.5)² = 25 + 20.25 = 45.25
G[1] += (0.15)² = 0.04 + 0.0225 = 0.0625

G = [45.25, 0.0625]
```

Effective learning rates:
```
lr_eff[0] = 0.1 / √45.25 = 0.1 / 6.73 ≈ 0.0149 (even smaller!)
lr_eff[1] = 0.1 / √0.0625 = 0.1 / 0.25 = 0.4 (slightly smaller)

Notice:
- w1 lr decreased further (because more gradient history)
- w2 lr decreased but less dramatically
```

Weight updates:
```
W_out[0] = 0.7 - 0.0149 × 4.5 ≈ 0.633
W_out[1] = 0.5 - 0.4 × 0.15 ≈ 0.44
```

**Pattern:**

```
w1 characteristics:
- Consistent large gradients
- Learning rate decreases over time
- Smaller and smaller updates

w2 characteristics:
- Consistent small gradients
- Learning rate stays moderate
- More updates toward this weight

This makes sense!
- Frequently updated weights: slow down (avoid overfitting)
- Infrequently updated weights: speed up (explore more)
```

### Adagrad Characteristics:

```
Pros:
✓ Adapts learning rate per weight
✓ Doesn't require learning rate tuning
✓ Good for sparse data (rare features get larger updates)
✓ Works well for convex problems

Cons:
✗ Learning rate monotonically decreases
✗ Eventually lr becomes too small (stops learning)
✗ Accumulation of G can cause problems in long training
✗ Doesn't do well on non-convex problems (deep learning)

The Achilles' Heel:
G is always increasing!
G += gradient² (never decreases)

Eventually:
lr_eff = lr / √G → 0

Weight updates become negligible!
Training gets stuck!

Example:
Iteration 100: G = 1000, lr_eff = 0.1 / √1000 ≈ 0.003
Iteration 1000: G = 10000, lr_eff = 0.1 / √10000 = 0.001
Iteration 10000: G = 100000, lr_eff = 0.1 / √100000 ≈ 0.0001

Learning rate decays too aggressively!
```

### When to Use Adagrad:

```
Use Adagrad when:
✓ Training on sparse data (NLP with embeddings)
✓ Each feature appears infrequently
✓ Convex problems (not deep neural networks)
✓ Training time is short

Don't use when:
✗ Training deep neural networks
✗ Need to train for many epochs
✗ Learning rate becomes too small

Modern: RMSProp and Adam are better (address decaying lr issue)
```

---

## 5. RMSProp (Root Mean Square Propagation)

### What is RMSProp?

RMSProp fixes Adagrad's problem of monotonically decreasing learning rate. It uses exponential moving average of squared gradients instead of accumulating all of them.

**Key Improvement:**
```
Adagrad problem:
G += gradient² (accumulates forever)
Eventually G so large that lr → 0

RMSProp solution:
Use exponential moving average
E[G] = β × E[G] + (1-β) × gradient²
(Forget old gradients, remember recent ones)

Effect:
Learning rate doesn't decay to zero
Can continue learning throughout training
```

### How RMSProp Works:

```
1. Keep exponential moving average of squared gradients: E[G]
2. Initialize E[G] = 0
3. For each iteration:
   - Compute gradient: g
   - Update E[G]: E[G] = β × E[G] + (1-β) × g²
   - Update weight: w = w - (lr / √(E[G] + ε)) × g

β: decay rate (typically 0.9)
ε: small constant (e.g., 1e-8)

Interpretation:
- E[G] is weighted average of recent squared gradients
- Older gradients fade out (×β each time)
- Recent gradients matter more
- Learning rate stays relatively constant
```

### RMSProp Example:

**Setup:**
```
Weight: w = 0.8
Learning rate: 0.1
Decay rate β: 0.9
Initial E[G] = 0
ε = 1e-8
```

**Iteration 1:**

```
Gradient: g = 5.0

E[G] = 0.9 × 0 + (1-0.9) × (5.0)²
     = 0 + 0.1 × 25
     = 2.5

Effective learning rate:
lr_eff = 0.1 / √(2.5 + 1e-8) ≈ 0.1 / 1.58 ≈ 0.063

w_new = 0.8 - 0.063 × 5.0
      = 0.8 - 0.315
      = 0.485
```

**Iteration 2:**

```
Gradient: g = 4.5

E[G] = 0.9 × 2.5 + (1-0.9) × (4.5)²
     = 2.25 + 0.1 × 20.25
     = 2.25 + 2.025
     = 4.275

Effective learning rate:
lr_eff = 0.1 / √4.275 ≈ 0.1 / 2.07 ≈ 0.048

w_new = 0.485 - 0.048 × 4.5
      = 0.485 - 0.216
      = 0.269
```

**Iteration 3:**

```
Gradient: g = 3.0

E[G] = 0.9 × 4.275 + (1-0.9) × (3.0)²
     = 3.848 + 0.1 × 9
     = 3.848 + 0.9
     = 4.748

Effective learning rate:
lr_eff = 0.1 / √4.748 ≈ 0.1 / 2.18 ≈ 0.046

w_new = 0.269 - 0.046 × 3.0
      = 0.269 - 0.138
      = 0.131
```

**Iteration 100 (suppose):**

```
Gradient: g = 0.5 (gradient decreases as approach minimum)

E[G] = 0.9 × 5.0 + (1-0.9) × (0.5)²
     = 4.5 + 0.025
     = 4.525

Effective learning rate:
lr_eff = 0.1 / √4.525 ≈ 0.047

w_new = current_w - 0.047 × 0.5

Even after 100 iterations:
Learning rate still ~0.047 (not decayed to zero!)

Compare to Adagrad:
After 100 iterations of similar gradients:
G would be huge (sum of all squared gradients)
Adagrad lr would be nearly zero
```

### Why RMSProp Works Better:

```
Comparison: Adagrad vs RMSProp

Iteration 1: 
  Adagrad G = 25, lr_eff = 0.02
  RMSProp E[G] = 2.5, lr_eff = 0.063

Iteration 10:
  Adagrad G ≈ 350, lr_eff ≈ 0.005
  RMSProp E[G] ≈ 4.5, lr_eff ≈ 0.047

Iteration 100:
  Adagrad G ≈ 2500, lr_eff ≈ 0.001
  RMSProp E[G] ≈ 5.0, lr_eff ≈ 0.045

Insight:
- Adagrad's lr decays from 0.02 to 0.001 (20x decrease)
- RMSProp's lr stays around 0.05 (stable)
- RMSProp continues learning, Adagrad stalls
```

### Decay Rate β Effect:

```
β = 0.9 (default):
E[G] = 0.9 × E[G] + 0.1 × g²
Effect: 90% old average, 10% new gradient
Remembers longer history

β = 0.99 (strong decay):
E[G] = 0.99 × E[G] + 0.01 × g²
Effect: 99% old average, 1% new gradient
Very smooth, slow adaptation

β = 0.5 (weak decay):
E[G] = 0.5 × E[G] + 0.5 × g²
Effect: equal old and new
Reacts quickly to changes

Typical: β = 0.9 or 0.999
```

### When to Use RMSProp:

```
Use RMSProp when:
✓ Training deep neural networks
✓ Need adaptive learning rate
✓ Want better than Adagrad (doesn't decay to zero)
✓ Non-convex problems

Better alternative: Adam (combines RMSProp + Momentum)
RMSProp less common now, but good choice!
```

---

## 6. Adam (Adaptive Moment Estimation) - THE INDUSTRY STANDARD

### What is Adam?

Adam combines the best of both worlds:
1. **Momentum**: Accumulates past gradients (velocity)
2. **RMSProp**: Adapts learning rate per weight (scale)

It's called "Adaptive Moment Estimation" because:
- First moment: mean (average) of gradients (momentum)
- Second moment: mean of squared gradients (scaling)

**Why Adam?**
```
Best parts of different optimizers:
- Momentum benefits: smooth convergence
- Adaptive learning rate: handles different scales
- Works for both convex and non-convex
- Robust across different problems
- Simple, few hyperparameters

Result: Industry standard for deep learning!
Default optimizer in most frameworks.
```

### How Adam Works:

```
Maintains two running averages:
1. m (first moment): exponential average of gradients
2. v (second moment): exponential average of squared gradients

Update rules:
m = β₁ × m + (1-β₁) × gradient
v = β₂ × v + (1-β₂) × gradient²
m_corrected = m / (1 - β₁^t)  [bias correction]
v_corrected = v / (1 - β₂^t)  [bias correction]
w = w - learning_rate × (m_corrected / (√v_corrected + ε))

Where:
β₁: decay rate for first moment (typically 0.9)
β₂: decay rate for second moment (typically 0.999)
t: iteration number
ε: small constant (typically 1e-8)
```

### Adam Example - Complete Training:

**Network Setup:**
```
Simple: 1 input → 1 hidden (2 neurons) → 1 output
Weights: W1 = [0.5, -0.3]ᵀ, W2 = [0.8, 0.6]
Input: x = 2.0, Target: y = 10.0

Adam Hyperparameters:
Learning rate: 0.001 (common value)
β₁ (momentum): 0.9
β₂ (RMSProp): 0.999
ε: 1e-8

Initialize:
m = 0 (momentum)
v = 0 (second moment)
```

**Iteration 1:**

Forward Pass:
```
z1 = [0.5, -0.3]ᵀ × 2 + [0, 0]ᵀ = [1.0, -0.6]ᵀ
a1 = ReLU([1.0, -0.6]ᵀ) = [1.0, 0]ᵀ
z2 = [0.8, 0.6] × [1.0, 0]ᵀ = 0.8
ŷ = 0.8

Loss = (10 - 0.8)² = 84.64
```

Backward Pass (Gradients):
```
∂Loss/∂W1 = [-16.8, 0]ᵀ (from chain rule)
(Let me compute just first element for clarity)

∂Loss/∂W1[0] = -16.8

(Second element is 0 because neuron was inactive)
```

Adam Update for W1[0]:

```
Step 1: Update momentum (m)
m = β₁ × m_old + (1-β₁) × gradient
m = 0.9 × 0 + (1-0.9) × (-16.8)
m = 0 + 0.1 × (-16.8)
m = -1.68

Step 2: Update second moment (v)
v = β₂ × v_old + (1-β₂) × gradient²
v = 0.999 × 0 + (1-0.999) × (-16.8)²
v = 0 + 0.001 × 282.24
v = 0.282

Step 3: Bias correction
m_corrected = m / (1 - β₁^t)
t = 1, β₁^1 = 0.9
m_corrected = -1.68 / (1 - 0.9)
m_corrected = -1.68 / 0.1
m_corrected = -16.8

v_corrected = v / (1 - β₂^t)
t = 1, β₂^1 = 0.999
v_corrected = 0.282 / (1 - 0.999)
v_corrected = 0.282 / 0.001
v_corrected = 282

Step 4: Update weight
w = w - lr × (m_corrected / (√v_corrected + ε))
w = 0.5 - 0.001 × (-16.8 / (√282 + 1e-8))
w = 0.5 - 0.001 × (-16.8 / 16.79)
w = 0.5 - 0.001 × (-1.0006)
w = 0.5 + 0.001
w = 0.501

Very small update! (That's why lr=0.001 for Adam)
```

**Iteration 2:**

Suppose new gradient: -14.5

```
Step 1: Update momentum
m = 0.9 × (-1.68) + (1-0.9) × (-14.5)
m = -1.512 - 1.45
m = -2.962

Momentum accumulates! Now -2.962 instead of -1.68

Step 2: Update second moment
v = 0.999 × 0.282 + (1-0.999) × (-14.5)²
v = 0.282 + 0.001 × 210.25
v = 0.282 + 0.210
v = 0.492

Second moment increasing (more gradient history)

Step 3: Bias correction
m_corrected = -2.962 / (1 - 0.9²)
m_corrected = -2.962 / (1 - 0.81)
m_corrected = -2.962 / 0.19
m_corrected = -15.59

v_corrected = 0.492 / (1 - 0.999²)
v_corrected = 0.492 / (1 - 0.998)
v_corrected = 0.492 / 0.002
v_corrected = 246

Step 4: Update weight
w = w - 0.001 × (-15.59 / (√246 + 1e-8))
w = w - 0.001 × (-15.59 / 15.68)
w = w - 0.001 × (-0.994)
w = w + 0.000994
w = 0.501 + 0.000994 ≈ 0.502
```

**Iteration 50:**

```
After many iterations (suppose):
m = -8.5 (accumulated momentum, negative direction)
v = 2.5 (accumulated squared gradients)
Gradient: -2.1 (smaller, approaching minimum)

m_corrected = -8.5 / (1 - 0.9^50) ≈ -8.5 / 1 ≈ -8.5
v_corrected = 2.5 / (1 - 0.999^50) ≈ 2.5 / 1 ≈ 2.5

w = w - 0.001 × (-8.5 / (√2.5 + ε))
w = w - 0.001 × (-8.5 / 1.58)
w = w - 0.001 × (-5.38)
w = w + 0.00538
```

### Why Adam Works So Well:

**Combining Forces:**

```
Momentum component (m):
✓ Smooths out oscillations
✓ Builds up velocity in consistent direction
✓ Accelerates through plateaus

Adaptive component (v):
✓ Weights with large gradients: smaller steps
✓ Weights with small gradients: larger steps
✓ Self-balancing per weight

Together:
✓ Smooth, fast convergence
✓ Adaptive per weight
✓ Combines best parts
✓ Works on complex problems
```

**Convergence Comparison:**

```
Loss over iterations:

Basic SGD:
Loss
  |*
  | *\*/*\/*
  |   \ / \ /
  |    X   X (oscillates)
  |________________ Iteration

Momentum:
Loss
  |*
  | **
  |  **
  |   *** (smoother)
  |_______

RMSProp:
Loss
  |*
  | **
  |  **
  |   *** (similar to momentum)
  |_______

Adam:
Loss
  |*
  | **
  |  **
  |   *** (similar, but more stable)
  |_______

All smooth, but Adam most robust across problems!
```

### Adam Hyperparameters:

```
Learning rate (lr): 0.001 (default, safe)
Range: 0.0001 to 0.01

β₁ (momentum): 0.9 (default)
Effect: Higher = more momentum
Range: 0.8 to 0.999

β₂ (second moment): 0.999 (default)
Effect: Higher = slower adaptation
Range: 0.99 to 0.9999

ε: 1e-8 (default)
Effect: Prevents division by zero
Don't change usually!

Typical settings:
```

### When to Use Adam:

```
Use Adam when:
✓ Default choice for deep learning!
✓ Training any neural network
✓ Don't want to tune learning rate much
✓ Need robust optimizer
✓ Working with non-convex problems
✓ Limited time/resources (good defaults)

Adam is safest choice!
Works well for 90% of problems.
```

---

## 7. Optimizer Comparison Summary

### Complete Comparison Table:

```
╔═══════════════════════════════════════════════════════════════════════╗
║ Optimizer    │ Adaptive │ Momentum │ Learning │ Convergence │ Use     ║
║              │ LR       │          │ Curve    │ Speed       │         ║
╠═══════════════════════════════════════════════════════════════════════╣
║ SGD          │ No       │ No       │ Oscillate│ Slow        │ Baseline║
║              │          │          │ wavy     │ (100 iter)  │         ║
║              │          │          │          │             │         ║
║ Momentum     │ No       │ Yes (0.9)│ Smoother │ Medium      │ Better  ║
║              │          │          │          │ (50 iter)   │ SGD     ║
║              │          │          │          │             │         ║
║ Nesterov     │ No       │ Yes+Look │ Smooth   │ Medium-Fast │ Fine-   ║
║              │          │ ahead    │          │ (50 iter)   │ tuned   ║
║              │          │          │          │             │         ║
║ Adagrad      │ Yes      │ No       │ Smooth   │ Fast then   │ Sparse  ║
║              │ (decays) │          │          │ stalls      │ data    ║
║              │          │          │          │             │         ║
║ RMSProp      │ Yes      │ No       │ Smooth   │ Fast        │ Good    ║
║              │(adaptive)│          │          │ (30 iter)   │ choice  ║
║              │          │          │          │             │         ║
║ Adam         │ Yes      │ Yes      │ Smooth   │ Very Fast   │ DEFAULT ║
║              │(adaptive)│ (0.9)    │          │ (20 iter)   │ CHOICE! ║
║              │          │          │          │             │         ║
╚═══════════════════════════════════════════════════════════════════════╝
```

### Visual Convergence Comparison:

```
Loss over 100 epochs (lower is better):

SGD:              Momentum:          Adam:
Loss              Loss               Loss
  100 *             100 *              100 *
   90 |*             90 |*              90 |*
   80 | *            80 | *             80 | *
   70 | *\           70 | **            70 | **
   60 |  *\          60 |  **           60 |  **
   50 |   *\         50 |   **          50 |   **
   40 |    *\        40 |    ***        40 |    ***
   30 |     *\       30 |       ***     30 |       ***
   20 |      *\      20 |            ** 20 |          **
   10 |       *\     10 |              *10 |           *
    0 |________\___  0 |________________ 0 |________________
      0        100        0        100        0        100

SGD: Oscillates, slow          Momentum: Smooth, better
                               Adam: Smoothest, fastest!
```

### Practical Recommendations:

```
For most problems (Deep Neural Networks):
→ Use Adam with default settings!
  learning_rate = 0.001
  β₁ = 0.9
  β₂ = 0.999

For simple/shallow models:
→ SGD with Momentum is fine
  learning_rate = 0.01 or 0.1

For sparse data (NLP, embeddings):
→ Adagrad or RMSProp
  (but Adam works too!)

For fine-tuning pre-trained models:
→ SGD with smaller learning rate
  learning_rate = 0.0001

If Adam not converging:
→ Lower learning rate (try 0.0001)
→ Or switch to SGD with momentum
```

---

## 8. Advanced: Learning Rate Scheduling

Optimizers can be combined with learning rate scheduling - changing learning rate during training.

### Why Learning Rate Scheduling?

```
Problem:
- Fixed learning rate not optimal throughout training
- Early: need large steps (far from optimum)
- Late: need small steps (near optimum, fine-tuning)

Solution:
- Start with larger learning rate
- Gradually decrease during training
- Smoother convergence

Analogy:
Finding hidden object in room
Early: walk fast (large area to search)
Later: walk slow (narrowing down)
Final: creep carefully (very close)
```

### Common Scheduling Strategies:

```
1. Step Decay
   Every N epochs: lr = lr × decay_rate
   
   Example: lr = 0.001, decay after 10 epochs
   Epoch 1-10: lr = 0.001
   Epoch 11-20: lr = 0.0001
   Epoch 21-30: lr = 0.00001

2. Exponential Decay
   lr = lr_0 × e^(-decay_rate × epoch)
   
   Smooth decrease (not step-wise)

3. Cosine Annealing
   lr = lr_min + 0.5 × (lr_max - lr_min) × (1 + cos(π × epoch/epochs))
   
   Smooth decrease shaped like cosine

4. Warmup then Decay
   Epoch 1-5: gradually increase lr
   Epoch 6+: gradually decrease lr
   
   Stabilizes early training
```

### Learning Rate Scheduling Example:

```
Training with Step Decay:

Epoch 1-10:
  lr = 0.001
  Loss: 2.5 → 0.5 (large decrease, large lr good)

Epoch 11-20:
  lr = 0.0001
  Loss: 0.5 → 0.1 (smaller decrease, smaller lr good)

Epoch 21-30:
  lr = 0.00001
  Loss: 0.1 → 0.08 (fine-tuning, tiny lr good)

Loss Curve:
Loss
  |*
  | \
  |  \*
  |   \
  |    *\
  |     \*
  |      \
  |       *\____
  |________________ Epoch
  
Steeper initially, then flattens!
Matches natural learning curve.
```

---

## Summary: Complete Optimizer Guide

### Decision Tree for Choosing Optimizer:

```
Starting a new project?
│
├─ Deep Neural Network (CNN, RNN, Transformer)?
│  └─ USE ADAM (default)
│     lr = 0.001
│     Usually just works!
│
├─ Simple/shallow model?
│  ├─ Want simplicity?
│  │  └─ USE SGD with Momentum
│  │     lr = 0.01 to 0.1
│  │
│  └─ Want adaptive learning rate?
│     └─ USE RMSProp
│        lr = 0.001
│
├─ Sparse data (embeddings, NLP)?
│  └─ Can use Adagrad
│     (or Adam, which works too)
│
├─ Fine-tuning pre-trained model?
│  └─ USE SGD with smaller lr
│     lr = 0.0001
│
└─ Not sure?
   └─ USE ADAM!
      (safe default, works for 90% of cases)
```

### Checklist for Using Optimizers:

```
1. Choose Optimizer
   ☐ Adam (default recommendation)
   ☐ Or SGD with Momentum
   ☐ Or RMSProp

2. Set Initial Learning Rate
   ☐ Adam: 0.001 (try this first)
   ☐ SGD+Momentum: 0.01 or 0.1
   ☐ RMSProp: 0.001

3. Train Model
   ☐ Monitor training loss
   ☐ Monitor validation loss
   ☐ Watch for convergence

4. If Not Converging
   ☐ Lower learning rate (×0.1)
   ☐ Train longer (more epochs)
   ☐ Check gradient magnitudes

5. If Diverging (loss increasing)
   ☐ Lower learning rate significantly
   ☐ Use gradient clipping
   ☐ Check data normalization

6. If Slow Convergence
   ☐ Increase learning rate (×2)
   ☐ Add momentum (SGD)
   ☐ Use learning rate scheduling
   ☐ Batch normalize

7. Once Converging Well
   ☐ Fine-tune hyperparameters
   ☐ Try learning rate scheduling
   ☐ Reduce learning rate near end
```

---

## The Complete Picture: From Theory to Practice

### A Realistic Training Scenario:

```
Start project: Training image classifier

Step 1: Initial Setup
Code:
from torch.optim import Adam
optimizer = Adam(model.parameters(), lr=0.001)

Step 2: First Training Run
Results: Loss decreasing smoothly → Good!
Converges in 50 epochs

Step 3: Validation Performance
Training accuracy: 95%
Validation accuracy: 85%
(Overfitting! But optimizer working)

Step 4: Improve Model
Add dropout
Increase regularization
(Not optimizer issue)

Step 5: Fine-tune Learning Rate
Try lr=0.0005: Slower convergence
Try lr=0.002: Slightly noisier, but similar
Keep lr=0.001 (default works!)

Step 6: Final Improvements
Add learning rate scheduling
After 30 epochs: lr = 0.0005
After 50 epochs: lr = 0.0001

Final results:
Converges faster
Smoother final convergence
Slightly better validation accuracy

Training complete!
```

The key insight: **Adam with lr=0.001 solves most problems. Only tune if necessary!**